# Experiment 01 — TFNO2D Baseline

Clean orchestration notebook. All reusable logic lives in `src/`.

**Pipeline:**  
1. Load config  
2. Compute normalization stats  
3. Build training/validation samples  
4. Train TFNO2D model  
5. Run inference and save `preds.npy`

In [ ]:
import sys, os

# ── Path resolution: works both locally and on Kaggle ──
# On Kaggle: attach your src dataset as input, set KAGGLE_SRC_DATASET below.
# Locally:   runs from notebooks/ so '../' resolves to Ronit/

KAGGLE_SRC_DATASET = "ronit-pm25-src"   # matches kushieboi/ronit-pm25-src

LOCAL_SRC  = os.path.abspath('../')
KAGGLE_SRC = f'/kaggle/input/{KAGGLE_SRC_DATASET}'

if os.path.exists('/kaggle'):
    # Running on Kaggle
    SRC_ROOT = KAGGLE_SRC
else:
    # Running locally
    SRC_ROOT = LOCAL_SRC

if SRC_ROOT not in sys.path:
    sys.path.insert(0, SRC_ROOT)

print(f"SRC_ROOT: {SRC_ROOT}")

from src.config import load_config
from src.utils import seed_everything, print_device_info, count_parameters
from src.data import compute_stats, save_stats, build_samples, get_dataloaders
from src.model import build_model
from src.train import train
from src.inference import run_inference

## 1. Configuration

In [ ]:
cfg = load_config()

seed_everything(cfg['training']['seed'])
print_device_info(cfg['device'])

print(f"Features ({cfg['features']['n_features']}): {cfg['features']['all']}")
print(f"Train months: {cfg['data']['train_months']}")
print(f"Val month:    {cfg['data']['val_month']}")

## 2. Compute Normalization Stats

In [ ]:
import os

stats = compute_stats(cfg, cfg['data']['train_months'])
save_stats(stats, os.path.join(cfg['paths']['temp'], 'norm_stats.npy'))
print('Stats computed for:', cfg['data']['train_months'])

## 3. Build Training & Validation Samples

In [ ]:
X_train, y_train = build_samples(
    cfg, cfg['data']['train_months'],
    stride=cfg['time']['stride_train'], stats=stats
)
X_val, y_val = build_samples(
    cfg, [cfg['data']['val_month']],
    stride=cfg['time']['stride_val'], stats=stats
)
print(f'Train: {X_train.shape}, Val: {X_val.shape}')

In [ ]:
train_dl, val_dl = get_dataloaders(cfg, X_train, y_train, X_val, y_val)

## 4. Build & Train Model

In [ ]:
model = build_model(cfg)
print(f'Parameters: {count_parameters(model):,}')

In [ ]:
history = train(cfg, model, train_dl, val_dl)

### Training Curves

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 4))
plt.plot(history['train_loss'], label='Train RMSE')
plt.plot(history['val_loss'], label='Val RMSE')
plt.xlabel('Epoch')
plt.ylabel('RMSE')
plt.legend()
plt.title('Training Curves')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 5. Inference & Submit

In [ ]:
import torch

model.load_state_dict(torch.load(cfg['paths']['model_save']))
preds = run_inference(cfg, model, stats)
print('Done!')